## Part 0: Environment Verification
* **Host Operating System:** macOS 26.6.1 (25G76)
* **AI Tooling Verification:** GitHub Copilot CLI authenticated and active

---

## Part 1: Basic Agent Operations & Security Policy Baseline
* **Prompt Submitted:** `Reply with exactly: Hello, CSCI 6032.`
* **Copilot Response:** `Hello, CSCI 6032.`
* **Tool / Permission Analysis:** The agent executed a basic text response. No host system permissions, file modifications, or credentials were requested or accessed.

---

## Part 2: Baseline Git Repository Checkpoint
* **Repository URL:** `https://github.com/mtwilson1/csci6032-hw2-mtwilson`
* **Baseline Commit Hash:** `fc44fc0`
* **Active Working Branch:** `agent-work`
* **Git Status:** `On branch agent-work, working tree clean`

---

## Part 3: Safety Governance & Sandbox Configuration
* **Policy File Created:** `AGENTS.md` (defining directory isolation, secret protection, and change transparency directives).
* **Environment Verification Command:** `/env`
* **Loaded Environment State:**
  ```text
  Custom Instructions
   - AGENTS.md

  MCP Servers
   - github-mcp-server (connected, builtin)

  Skills
   - customize-cloud-agent (Built-in) - /Users/[REDACTED]/Library/Caches/copilot/pkg/darwin-arm64/1.0.83/builtin/customize-cloud-agent/SKILL.md
   - github-pr-media (Built-in) - /Users/[REDACTED]/Library/Caches/copilot/pkg/darwin-arm64/1.0.83/builtin/github-pr-media/SKILL.md

  Custom Agents
  No custom agents loaded.

  Plugins
  No plugins installed.

  Hooks
  No hooks loaded.

  Extensions
  No extensions loaded.
  ```
* **Security Baseline Analysis:** Copilot CLI automatically detected and loaded `AGENTS.md` under Custom Instructions, confirming active governance enforcement for tool permissions and directory boundaries.

---

## Part 4: Python Utility Development & Tool Output Verification
* **Prompt Submitted to Agent:** `Create a Python script named text_stats.py that takes a file path as a command-line argument, reads the file, and prints the total number of lines, words, and characters. Handle file not found errors gracefully.`
* **Utility Script Created:** `text_stats.py`
* **Test Command (Valid File):** `python3 text_stats.py sample.txt`
* **Utility Output:**
  ```text
  Lines: 5
  Words: 44
  Characters: 235
  ```
* **Test Command (Missing File):** `python3 text_stats.py does-not-exist.txt`
* **Utility Output:**
  ```text
  Error: file not found: does-not-exist.txt
  ```
* **Verification Analysis:** The agent successfully created `text_stats.py` using standard argument parsing and error handling. Output verified line, word, and character counts against `sample.txt`, and gracefully handled nonexistent paths without tracebacks.

**Record the exact prompt you used:**
"Refactor our project structure for Part 4: 1. Move text_stats.py into src/text_stats.py. 2. Update src/text_stats.py so that it outputs valid JSON with keys: lines, words, characters. 3. Create unit tests using the unittest framework in tests/test_text_stats.py. 4. Update README.md with usage documentation. 5. Run the script on sample.txt to verify."

**Include selected agent output:**
Done.
• Moved the script to src/text_stats.py
• Updated it to print valid JSON with keys: lines, words, characters
• Added tests/test_text_stats.py using Python’s built-in unittest
• Updated README.md with usage docs
• Verified with sample.txt
Sample output: {"lines": 5, "words": 44, "characters": 235}
Validation: Passed.

**Record the test command and result:**
Command: `python3 src/text_stats.py sample.txt && python3 -m unittest discover -s tests -v`
Result: The script successfully output `{"lines": 5, "words": 44, "characters": 235}` and the unit tests passed without errors.

**The independent check you performed:**
I manually verified the contents of `sample.txt` to confirm it contains exactly 5 lines, 44 words, and 235 characters, proving the script's JSON output is accurate.

**What did you inspect before deciding that the change was safe to commit? Did the agent do anything unexpected?**
Inspection: I verified that every file operation prompt was restricted to the `~/Desktop/csci6032-hw2-mtwilson/` directory, adhering strictly to the privacy and security boundaries established in `AGENTS.md`. I also reviewed the generated Python files to ensure no unauthorized files were accessed.

Unexpected Actions: The agent initially hallucinated incorrect word and character counts (33 words, 229 characters) when writing the tests and documentation. When it executed the script and got the real values, it autonomously went back and corrected `test_text_stats.py` and `README.md`. It also proactively changed its execution commands from `python` to `python3` for macOS compatibility.

**Commit Identifier:**
[b94622c4a78b6bc4b9b890fac740b0666ffe7771]

### Part 5: Install and verify Docker on the host

**Record the Docker product, installation method, version, and a concise excerpt showing that `hello-world` ran:**
* **Docker Product:** Docker Desktop for Mac
* **Installation Method:** Downloaded the official installer from the Docker website.
* **Docker Version Output:**
Client:
 Version:           29.8.0
 OS/Arch:           darwin/arm64
Server: Docker Desktop 4.91.0 (239619)
 Engine Version:    29.8.0
 OS/Arch:           linux/arm64

* **Hello-World Excerpt:**
Hello from Docker!
This message shows that your installation appears to be working correctly.

**Note any virtualization, WSL, permission, or architecture issue you encountered:**
I am running macOS on Apple Silicon (arm64). I initially encountered a `command not found: docker` error because my terminal was opened before Docker Desktop finished its first-time setup. Once I started the engine and opened a fresh terminal window to refresh my system path, everything ran perfectly without any virtualization or permission issues.

### Part 6: Secure Container Environment Setup

**1. Container Configuration**
Using the GitHub Copilot CLI, I generated a secure `Dockerfile` and `.dockerignore` in the root of the repository. The container was built using `python:3.12-slim` as the base image. To satisfy security constraints, I created a non-root user (`agentuser`), set the working directory to `/workspace`, and ensured no `COPY` commands were used to prevent hardcoding local files into the image. 

**2. Toolchain Installation**
The container was provisioned with the following tools via the `Dockerfile`:
* `git` (v2.47.3)
* `gh` (GitHub CLI v2.46.0)
* `nodejs` and `npm`
* `@github/copilot` (GitHub Copilot CLI v1.0.84, installed globally via npm)

**3. Execution and Isolation**
The image was built and tagged as `csci6032-hw2-agent`. The container was instantiated using an interactive shell and a strict bind mount linking the host's repository folder to the container's `/workspace`. This ensures the environment remains isolated from the host machine.

**4. Authentication**
Due to the headless nature of the Docker container, both the GitHub CLI and Copilot CLI were authenticated using the device code flow (`gh auth login` and `copilot login --device-code`). Verification was confirmed by successfully querying the repository metadata via `gh repo view`.

## Part 7: Containerized Agent Execution and Feature Implementation

**Prompt Submitted to Container Agent:**
`Extend src/text_stats.py with an optional --top N argument that reports the N most frequent words, case-insensitively, with deterministic tie-breaking. Update the unittest tests and README.md. Do not add external dependencies. Do not commit or push. First inspect the existing project and explain your plan. After editing, show the diff and run all tests.`

**Evidence of container execution and tests:**
The agent executed the modifications and tests successfully inside the container on the bind-mounted files. Manual verification on the host Mac produced the expected JSON output including the top 3 words:

```text
----------------------------------------------------------------------
Ran 6 tests in 0.255s

OK
{"lines": 5, "words": 44, "characters": 235, "top_words": [{"word": "the", "count": 4}, {"word": "this", "count": 4}, {"word": "file", "count": 3}]}

## Part 8: Add browser tools on the host

**Record the extension name and source, the location of the MCP configuration, and the result of the read-only browser test:**
* **Extension Name & Source:** Playwright MCP Bridge extension, installed from the instructor-approved link.
* **MCP Configuration Location:** Project-level MCP configuration file.
* **Read-only Browser Test Result:** I asked the agent to report the title of my currently active tab. The agent successfully connected through the extension's connection dialog, read the tab data securely, and correctly reported the title as "Institution Page" without taking any unauthorized actions.
**What additional authority did connecting the browser give the agent?**
Connecting the browser granted the agent the authority to view, read, and interact with the Document Object Model (DOM) of the specifically allowed Blackboard tab. Because the browser is already authenticated, this allows the agent to leverage my existing session cookies to navigate pages or simulate clicks on my behalf within the boundaries of that approved tab.


## Part 9: Create a Blackboard submission skill

**Paste your complete SKILL.md here as a fenced code block.**
```yaml
---
name: blackboard-submission
description: A workflow skill to preflight, package, and safely submit CSCI 6032 Homework 2 to Blackboard.
---

# Blackboard Submission Protocol

Follow this workflow strictly. Do NOT auto-approve terminal commands or browser actions.

## 1. Preflight and Validation
- Confirm you are operating in the expected homework repository (`csci6032-hw2-mtwilson`).
- Run a preflight check: check the current branch, run `git status`, check recent commits, and verify the expected remote URL using `git remote -v`.
- Stop immediately if the tree is not clean, required artifacts are missing, or unresolved secrets/private data are apparent.
- Confirm that the current reviewed branch has been pushed.

## 2. Packaging
- Prepare `csci6032-hw2-mtwilson.tar.gz` from the committed `HEAD`. 
- EXCLUDE `.git`, credentials, caches, or unrelated files.
- List the archive contents and show the exact notebook, archive, repository URL, and submission text you propose to use.

## 3. Dry Run Support
- Support a "dry run" mode. If the user requests a dry run, perform every possible check but STOP before opening Blackboard or submitting.

## 4. Browser Navigation and Authentication
- Ask the user before opening or controlling the Blackboard tab.
- Require the user to authenticate personally. NEVER request, read, store, type, or expose credentials.
- Navigate only to this homework's submission page and stage the required files and repository URL.

## 5. Final Irreversible Submission
- STOP IMMEDIATELY before the final, irreversible submission action (clicking submit).
- Show the user exactly what will be submitted.
- Require explicit confirmation at that point. A prior general approval is not sufficient.
- After confirmation, complete the submission, verify the confirmation page or receipt, and report the result.
- Do not commit Blackboard screenshots, receipts, browser data, or personal information to the public repository.
